In [12]:
import torch
import torch.nn as nn
from torchvision import models, transforms

class MultiTaskEfficientNet(nn.Module):
    def __init__(self, num_disease_classes):
        super(MultiTaskEfficientNet, self).__init__()
        self.backbone = models.efficientnet_b0()
        feature_dim = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Identity() 

        self.disease_head = nn.Sequential(
            nn.Linear(feature_dim, 512), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(512, num_disease_classes)
        )
        self.skin_head = nn.Sequential(
            nn.Linear(feature_dim, 256), nn.ReLU(),
            nn.Linear(256, 6) 
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.disease_head(features), self.skin_head(features)

MODEL_PATH = r'X:\Python\ML\Projects\Skin_disease\Skin_Disease_Prediction\backend\models\skin_disease_multitask_model.pth'
NUM_CLASSES = 114
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MultiTaskEfficientNet(NUM_CLASSES)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.to(DEVICE).eval()

# Load your model structure
# model = MultiTaskEfficientNet(num_classes=114)
# model.load_state_dict(torch.load("backend\models\skin_disease_multitask_model.pth", map_location="cpu"))
# model.eval()

# Create dummy input (Batch, Channel, Height, Width)
dummy_input = torch.randn(1, 3, 224, 224)

# Export to ONNX
torch.onnx.export(
    model, 
    dummy_input, 
    "model.onnx", # Save this directly into your frontend/public folder
    export_params=True,
    opset_version=12,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['disease_output', 'skin_output']
)
print("✅ Model converted to ONNX!")

✅ Model converted to ONNX!
